Fabric notebook: Generate Synthetic Event Streams → Eventhouse (Direct Ingest)

This notebook reads from the lh_insurance Lakehouse tables and generates
realistic event streams, ingesting them directly into Eventhouse KQL
tables using the Kusto Python SDK.

Event types generated:
  1. ClaimStatusEvent    — Claim lifecycle transitions (initial_review,
     site_inspection, document_request, estimate_prepared, payment_issued,
     denial_issued)
  2. FraudAlertEvent     — Automated fraud detection alerts on suspicious claims
  3. InspectionEvent     — Asset inspection scheduling and results
  4. PolicyChangeEvent   — Policy modifications (endorsements, renewals, cancellations)
  5. PaymentEvent        — Claim payment processing and disbursements

Demo scenarios injected:
  - Late Resolution: claim processing falling behind SLA targets
  - Major Loss: severe incident reported during active policy period
  - Adjuster Overload Risk: adjuster approaching maximum caseload limits
  - Inspection Due: scheduled asset inspection coming due per policy terms
  - Claim Reassignment: claim reassigned to specialist adjuster after major loss

Prerequisites:
  - Attach to "lh_insurance" Lakehouse
  - Run 01_load_reference_data notebook first
  - Create Eventhouse tables using schemas/eventhouse_setup.kql
  - Set KUSTO_URI and KUSTO_DATABASE below

To run: Execute all cells in order.


<h1 align="center">Insurance Claims Intelligence Workshop</h1>
<h3 align="center">Notebook 02 — Generate Simulated Events</h3>

---

> **Purpose:** This notebook generates simulated insurance lifecycle events for
> demonstration and testing purposes. It produces realistic claim processing
> activity — status transitions, fraud alerts, inspections, policy changes,
> and payments — and ingests them into an Eventhouse for real-time querying
> alongside the ontology and semantic model.
>
> **Simulation model:** Each active claim is tracked through its lifecycle.
> Adjusters process claims, inspections are scheduled and completed, payments
> are issued, and fraud detection runs continuously. Demo scenarios inject
> realistic operational stress patterns (SLA breaches, adjuster overload, etc.).

In [3]:
print(f"✓ Eventhouse URI: {KUSTO_URI}")
print(f"✓ KQL Database: {KUSTO_DATABASE}")
print(f"✓ Ingest URI: {INGEST_URI}")

StatementMeta(, 6a49e9da-8925-4fa1-9ba4-f7cf5e57595f, 5, Finished, Available, Finished, False)

AttributeError: module 'notebookutils' has no attribute 'widgets'

## Install Dependencies

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "azure-kusto-data",
                       "azure-kusto-ingest",
                       "azure-identity==1.16.0",
                       "requests>=2.32.3",
                       "--quiet"])

## Configuration

In [ ]:
import json
import uuid
import random
import time
from datetime import datetime, timedelta

# Simulation parameters
SIMULATION_DURATION_MINUTES = 1      # How long to run the simulation
TICK_INTERVAL_SECONDS = 10           # Processing cycle frequency
SCENARIO_INJECTION_ENABLED = True    # Inject demo scenarios
BATCH_SIZE = 100                     # Rows per ingest batch (Kusto streaming)
PRINT_EVENTS = True                  # Print events to console

# Claim processing timing (average seconds per status transition)
AVG_REVIEW_TIME_SECONDS = 40
AVG_INSPECTION_TIME_SECONDS = 60
AVG_DOCUMENT_TIME_SECONDS = 30
AVG_ESTIMATE_TIME_SECONDS = 40

random.seed(None)  # Use current time for varied runs

## Connect to Eventhouse (Kusto)

In [ ]:
from azure.kusto.data import KustoClient, KustoConnectionStringBuilder, DataFormat
from azure.kusto.ingest import QueuedIngestClient, IngestionProperties
import io
import pandas as pd

# Fabric notebooks use mssparkutils for authentication
KUSTO_TOKEN_SCOPE = "https://kusto.kusto.windows.net"

def get_fabric_token():
    """Get an access token using Fabric's built-in credential provider."""
    return mssparkutils.credentials.getToken(KUSTO_TOKEN_SCOPE)

print(f"✓ Query URI : {KUSTO_URI}")
print(f"✓ Ingest URI: {INGEST_URI}")
print(f"✓ Database  : {KUSTO_DATABASE}")

# Query client
kcsb = KustoConnectionStringBuilder.with_token_provider(
    KUSTO_URI,
    get_fabric_token
)
kusto_client = KustoClient(kcsb)

# Ingest client - use the real ingestion URI passed from the parent notebook
ingest_kcsb = KustoConnectionStringBuilder.with_token_provider(
    INGEST_URI,
    get_fabric_token
)
ingest_client = QueuedIngestClient(ingest_kcsb)

# Verify query connection
result = kusto_client.execute(KUSTO_DATABASE, ".show database schema | count")
for row in result.primary_results[0]:
    print(f"✓ Connected to Eventhouse query endpoint")
    print(f"  Database: {KUSTO_DATABASE}")

## Ingestion Helper

In [ ]:
# Column order for each table (must match the .create-merge-table schema)
TABLE_COLUMNS = {
    "ClaimStatusEvent": [
        "event_id", "event_type", "timestamp", "source",
        "claim_id", "claim_number", "policy_id", "adjuster_id",
        "previous_status", "new_status", "priority",
        "incident_latitude", "incident_longitude", "notes",
    ],
    "FraudAlertEvent": [
        "event_id", "event_type", "timestamp", "source",
        "claim_id", "policy_id", "policyholder_id",
        "alert_type", "severity", "confidence_score",
        "description", "recommended_action",
    ],
    "InspectionEvent": [
        "event_id", "event_type", "timestamp", "source",
        "claim_id", "asset_id", "adjuster_id",
        "inspection_type", "result", "damage_estimate",
        "latitude", "longitude", "notes",
    ],
    "PolicyChangeEvent": [
        "event_id", "event_type", "timestamp", "source",
        "policy_id", "policyholder_id",
        "change_type", "previous_value", "new_value",
        "effective_date", "notes",
    ],
    "PaymentEvent": [
        "event_id", "event_type", "timestamp", "source",
        "claim_id", "policy_id", "payee_id",
        "payment_type", "amount", "currency",
        "payment_status", "payment_method", "notes",
    ],
}

# Buffers: collect rows per table, flush when BATCH_SIZE is reached
_event_buffers = {table: [] for table in TABLE_COLUMNS}
_ingest_counts = {table: 0 for table in TABLE_COLUMNS}

def flatten_event(event):
    """Flatten the envelope + body into a single dict for tabular ingest."""
    row = {
        "event_id": event["event_id"],
        "event_type": event["event_type"],
        "timestamp": event["timestamp"],
        "source": event["source"],
    }
    row.update(event["body"])
    return row

def buffer_event(event):
    """Add event to the appropriate table buffer; flush if full."""
    table = event["event_type"]
    row = flatten_event(event)
    _event_buffers[table].append(row)

    if len(_event_buffers[table]) >= BATCH_SIZE:
        flush_buffer(table)

def flush_buffer(table):
    """Ingest buffered rows into the Eventhouse table."""
    rows = _event_buffers[table]
    if not rows:
        return

    columns = TABLE_COLUMNS[table]
    df = pd.DataFrame(rows, columns=columns)

    props = IngestionProperties(
        database=KUSTO_DATABASE,
        table=table,
        data_format=DataFormat.CSV,
    )

    csv_buffer = io.StringIO()
    df.to_csv(csv_buffer, index=False, header=False)
    csv_buffer.seek(0)
    csv_bytes = io.BytesIO(csv_buffer.getvalue().encode("utf-8"))

    ingest_client.ingest_from_stream(csv_bytes, ingestion_properties=props)
    _ingest_counts[table] += len(rows)
    _event_buffers[table] = []

def flush_all_buffers():
    """Flush all remaining buffered events."""
    for table in TABLE_COLUMNS:
        flush_buffer(table)

## Load Reference Data from Lakehouse

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Load all reference tables (fully-qualified so no default lakehouse context is needed)
lakehouse_path = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}/Tables"
offices_df       = spark.read.format("delta").load(f"{lakehouse_path}/offices").toPandas()
policyholders_df = spark.read.format("delta").load(f"{lakehouse_path}/policyholders").toPandas()
insured_assets_df = spark.read.format("delta").load(f"{lakehouse_path}/insured_assets").toPandas()
adjusters_df     = spark.read.format("delta").load(f"{lakehouse_path}/adjusters").toPandas()
policies_df      = spark.read.format("delta").load(f"{lakehouse_path}/policies").toPandas()
claims_df        = spark.read.format("delta").load(f"{lakehouse_path}/claims").toPandas()
claim_events_df  = spark.read.format("delta").load(f"{lakehouse_path}/claim_events").toPandas()
inspections_df   = spark.read.format("delta").load(f"{lakehouse_path}/asset_inspections").toPandas()

# Build lookup dictionaries
offices       = {row["office_id"]: row.to_dict() for _, row in offices_df.iterrows()}
policyholders = {row["policyholder_id"]: row.to_dict() for _, row in policyholders_df.iterrows()}
insured_assets = {row["asset_id"]: row.to_dict() for _, row in insured_assets_df.iterrows()}
adjusters     = {row["adjuster_id"]: row.to_dict() for _, row in adjusters_df.iterrows()}
policies      = {row["policy_id"]: row.to_dict() for _, row in policies_df.iterrows()}
claims        = {row["claim_id"]: row.to_dict() for _, row in claims_df.iterrows()}

# Get active claims for simulation using the insurance demo statuses
active_claims = claims_df[
    claims_df["status"].isin(["filed", "under_review", "investigation", "approved"])
].to_dict("records")

print(f"Loaded {len(active_claims)} active claims for simulation")
print(f"Reference data: {len(offices)} offices, {len(policyholders)} policyholders, "
      f"{len(adjusters)} adjusters, {len(policies)} policies")

## Helper Functions

In [ ]:
def new_event_id():
    return str(uuid.uuid4())

def make_envelope(event_type, source, body):
    """Wrap event body in standard envelope."""
    return {
        "event_id": new_event_id(),
        "event_type": event_type,
        "timestamp": datetime.utcnow().isoformat() + "Z",
        "source": source,
        "body": body,
    }

# Claim lifecycle: valid status transitions
CLAIM_STATUS_FLOW = [
    "initial_review",
    "site_inspection",
    "document_request",
    "estimate_prepared",
    "payment_issued",   # or denial_issued (branching)
]

FRAUD_ALERT_TYPES = [
    {"alert_type": "duplicate_claim",   "description": "Potential duplicate claim detected — similar incident details found on another policy",     "severity": "high"},
    {"alert_type": "excessive_amount",  "description": "Claimed loss amount significantly exceeds asset valuation",                                 "severity": "high"},
    {"alert_type": "recent_policy",     "description": "Claim filed within 90 days of policy inception — elevated risk indicator",                  "severity": "medium"},
    {"alert_type": "frequency_spike",   "description": "Policyholder claim frequency exceeds historical baseline",                                 "severity": "medium"},
    {"alert_type": "address_mismatch",  "description": "Incident location inconsistent with policyholder or asset registered address",              "severity": "low"},
    {"alert_type": "late_reporting",    "description": "Significant delay between incident date and claim filing date",                             "severity": "low"},
]

INSPECTION_TYPES = ["field_inspection", "desk_review", "photo_appraisal", "independent_appraisal"]

PAYMENT_METHODS = ["ACH", "check", "wire_transfer"]

## Claim Simulation State

In [ ]:
class ClaimSimState:
    """Tracks the simulated processing state of an active insurance claim."""

    def __init__(self, claim, policy, policyholder, asset, adjuster, office):
        self.claim = claim
        self.policy = policy
        self.policyholder = policyholder
        self.asset = asset
        self.adjuster = adjuster
        self.office = office

        # Current position in the claim lifecycle
        self.current_status = "initial_review"
        self.status_index = 0
        self.ticks_in_status = 0
        self.ticks_to_advance = random.randint(2, 6)  # ticks before next transition

        # Timing
        filed_str = claim.get("filed_date") or claim.get("incident_date")
        try:
            self.filed_time = datetime.fromisoformat(filed_str.replace("Z", ""))
        except (ValueError, AttributeError, TypeError):
            self.filed_time = datetime.utcnow() - timedelta(days=random.randint(1, 30))

        # Claim attributes
        self.estimated_loss = float(claim.get("estimated_loss") or random.uniform(5000, 150000))
        self.priority = claim.get("priority", "medium")
        self.incident_lat = claim.get("incident_latitude") or office.get("latitude", 0)
        self.incident_lon = claim.get("incident_longitude") or office.get("longitude", 0)

        # Scenario injection flags
        self.late_resolution_injected = False
        self.major_loss_injected = False
        self.overload_injected = False
        self.inspection_due_injected = False
        self.reassignment_injected = False

        # Track whether claim is finalized
        self.finalized = False

    def advance_tick(self):
        """Advance one simulation tick. Returns True if status transitioned."""
        if self.finalized:
            return False

        self.ticks_in_status += 1

        if self.ticks_in_status >= self.ticks_to_advance:
            return self._transition_status()
        return False

    def _transition_status(self):
        """Move claim to next lifecycle status."""
        self.status_index += 1
        if self.status_index >= len(CLAIM_STATUS_FLOW):
            # Final status: randomly choose payment or denial
            if random.random() < 0.15:
                self.current_status = "denial_issued"
            else:
                self.current_status = "payment_issued"
            self.finalized = True
        else:
            self.current_status = CLAIM_STATUS_FLOW[self.status_index]

        self.ticks_in_status = 0
        self.ticks_to_advance = random.randint(2, 6)
        return True

    @property
    def previous_status(self):
        if self.status_index == 0:
            return "open"
        if self.status_index > 0 and self.status_index <= len(CLAIM_STATUS_FLOW):
            return CLAIM_STATUS_FLOW[self.status_index - 1]
        return CLAIM_STATUS_FLOW[-2]

    @property
    def days_since_filed(self):
        return (datetime.utcnow() - self.filed_time).days

    @property
    def is_high_value(self):
        return self.estimated_loss > 100000

## Event Generators

In [ ]:
def generate_claim_status_event(state, notes=""):
    """Generate a ClaimStatusEvent for a claim lifecycle transition."""
    return make_envelope("ClaimStatusEvent", "claims-system", {
        "claim_id": state.claim["claim_id"],
        "claim_number": state.claim["claim_number"],
        "policy_id": state.claim["policy_id"],
        "adjuster_id": state.claim["adjuster_id"],
        "previous_status": state.previous_status,
        "new_status": state.current_status,
        "priority": state.priority,
        "incident_latitude": state.incident_lat,
        "incident_longitude": state.incident_lon,
        "notes": notes or f"Claim transitioned to {state.current_status}",
    })


def generate_fraud_alert_event(state, alert=None):
    """Generate a FraudAlertEvent for a suspicious claim."""
    if alert is None:
        alert = random.choice(FRAUD_ALERT_TYPES)
    return make_envelope("FraudAlertEvent", "fraud-detection-engine", {
        "claim_id": state.claim["claim_id"],
        "policy_id": state.claim["policy_id"],
        "policyholder_id": state.claim["policyholder_id"],
        "alert_type": alert["alert_type"],
        "severity": alert["severity"],
        "confidence_score": round(random.uniform(0.55, 0.98), 2),
        "description": alert["description"],
        "recommended_action": "refer_to_siu" if alert["severity"] == "high" else "flag_for_review",
    })


def generate_inspection_event(state, inspection_type=None, result=None):
    """Generate an InspectionEvent for an asset inspection."""
    if inspection_type is None:
        inspection_type = random.choice(INSPECTION_TYPES)
    if result is None:
        result = random.choice(["completed", "scheduled", "in_progress"])

    damage_estimate = round(state.estimated_loss * random.uniform(0.6, 1.1), 2) if result == "completed" else None

    return make_envelope("InspectionEvent", "inspection-service", {
        "claim_id": state.claim["claim_id"],
        "asset_id": state.claim["asset_id"],
        "adjuster_id": state.claim["adjuster_id"],
        "inspection_type": inspection_type,
        "result": result,
        "damage_estimate": damage_estimate,
        "latitude": state.incident_lat,
        "longitude": state.incident_lon,
        "notes": f"{inspection_type.replace('_', ' ').title()} — {result}",
    })


def generate_policy_change_event(state, change_type=None):
    """Generate a PolicyChangeEvent for a policy modification."""
    if change_type is None:
        change_type = random.choice(["endorsement_added", "coverage_updated", "deductible_changed", "renewal_processed"])

    policy = state.policy
    prev_val, new_val = "", ""
    if change_type == "deductible_changed":
        prev_val = str(policy.get("deductible", 1000))
        new_val = str(float(prev_val) + random.choice([-500, 500, 1000]))
    elif change_type == "coverage_updated":
        prev_val = str(policy.get("coverage_amount", 100000))
        new_val = str(float(prev_val) * random.choice([1.1, 1.25, 0.9]))
    else:
        prev_val = "N/A"
        new_val = change_type.replace("_", " ").title()

    return make_envelope("PolicyChangeEvent", "policy-admin-system", {
        "policy_id": state.claim["policy_id"],
        "policyholder_id": state.claim["policyholder_id"],
        "change_type": change_type,
        "previous_value": prev_val,
        "new_value": new_val,
        "effective_date": datetime.utcnow().isoformat() + "Z",
        "notes": f"Automated {change_type.replace('_', ' ')} during claim processing",
    })


def generate_payment_event(state, payment_type="claim_settlement"):
    """Generate a PaymentEvent for a claim payment."""
    amount = round(state.estimated_loss * random.uniform(0.5, 0.95), 2)
    return make_envelope("PaymentEvent", "payment-processing", {
        "claim_id": state.claim["claim_id"],
        "policy_id": state.claim["policy_id"],
        "payee_id": state.claim["policyholder_id"],
        "payment_type": payment_type,
        "amount": amount,
        "currency": "USD",
        "payment_status": "processed",
        "payment_method": random.choice(PAYMENT_METHODS),
        "notes": f"Settlement payment for claim {state.claim['claim_number']}",
    })

## Scenario Injectors

In [ ]:
def check_and_inject_scenarios(state, events):
    """
    Check if any demo scenario conditions are met and inject events.
    Modifies state flags and appends to events list.
    """
    claim = state.claim

    # --- SCENARIO 1: Late Resolution ---
    # Claim has been in processing too long relative to its priority
    if not state.late_resolution_injected and state.days_since_filed > 14:
        sla_days = {"high": 7, "medium": 14, "low": 30}.get(state.priority, 14)
        if state.days_since_filed > sla_days and not state.finalized:
            state.late_resolution_injected = True
            policyholder = state.policyholder
            print(f"\n  SCENARIO: Late Resolution — Claim {claim['claim_number']}")
            print(f"     Days since filed: {state.days_since_filed}, SLA: {sla_days} days")
            print(f"     Policyholder: {policyholder.get('full_name', 'Unknown')} ({policyholder.get('email', 'N/A')})")
            events.append(generate_claim_status_event(
                state, notes=f"ALERT: SLA breach — claim open {state.days_since_filed} days (target: {sla_days})"
            ))

    # --- SCENARIO 2: Major Loss ---
    # High-value claim triggers fraud screening and priority escalation
    if (not state.major_loss_injected
            and state.is_high_value
            and state.current_status in ("initial_review", "site_inspection")):
        state.major_loss_injected = True
        state.priority = "high"
        high_alert = [a for a in FRAUD_ALERT_TYPES if a["severity"] == "high"]
        print(f"\n  SCENARIO: Major Loss — Claim {claim['claim_number']}")
        print(f"     Estimated loss: ${state.estimated_loss:,.2f}")
        print(f"     Escalated to high priority, fraud screening triggered")
        events.append(generate_fraud_alert_event(state, random.choice(high_alert)))
        events.append(generate_claim_status_event(
            state, notes=f"ALERT: Major loss (${state.estimated_loss:,.0f}) — escalated to high priority"
        ))

    # --- SCENARIO 3: Adjuster Overload Risk ---
    # Count how many active claims this adjuster has
    if not state.overload_injected:
        adjuster_id = claim["adjuster_id"]
        adjuster = state.adjuster
        max_claims = int(adjuster.get("max_active_claims", 15))
        adjuster_claim_count = sum(
            1 for c in active_claims if c["adjuster_id"] == adjuster_id
        )
        if adjuster_claim_count >= max_claims - 2:
            state.overload_injected = True
            print(f"\n  SCENARIO: Adjuster Overload Risk — {adjuster.get('first_name', '')} {adjuster.get('last_name', '')}")
            print(f"     Active claims: {adjuster_claim_count}, Max capacity: {max_claims}")
            print(f"     Supervisor: {adjuster.get('supervisor_email', 'N/A')}")
            events.append(generate_claim_status_event(
                state, notes=f"ALERT: Adjuster at {adjuster_claim_count}/{max_claims} capacity — reassignment recommended"
            ))

    # --- SCENARIO 4: Inspection Due ---
    # When claim reaches site_inspection, generate an inspection event
    if (not state.inspection_due_injected
            and state.current_status == "site_inspection"):
        state.inspection_due_injected = True
        asset = state.asset
        last_inspection = asset.get("last_inspection_date", "N/A")
        print(f"\n  SCENARIO: Inspection Due — Asset {asset.get('asset_number', 'Unknown')}")
        print(f"     Type: {asset.get('asset_type', 'Unknown')} — {asset.get('make', '')} {asset.get('model', '')}")
        print(f"     Last inspection: {last_inspection}")
        events.append(generate_inspection_event(state, "field_inspection", "scheduled"))

    # --- SCENARIO 5: Claim Reassignment ---
    # After major loss, reassign to a specialist adjuster
    if (state.major_loss_injected
            and not state.reassignment_injected
            and state.current_status in ("document_request", "estimate_prepared")):
        state.reassignment_injected = True
        current_adjuster = state.adjuster
        claim_type = claim.get("claim_type", "general")

        # Find a specialist adjuster with capacity
        eligible_adjusters = [
            a for a in adjusters.values()
            if a["adjuster_id"] != claim["adjuster_id"]
            and a.get("status") == "active"
            and claim_type in (a.get("specializations") or [])
        ]
        # Fallback: any active adjuster with capacity
        if not eligible_adjusters:
            eligible_adjusters = [
                a for a in adjusters.values()
                if a["adjuster_id"] != claim["adjuster_id"]
                and a.get("status") == "active"
            ]

        print(f"\n  SCENARIO: Claim Reassignment — Claim {claim['claim_number']}")
        print(f"     From: {current_adjuster.get('first_name', '')} {current_adjuster.get('last_name', '')}")
        if eligible_adjusters:
            new_adj = random.choice(eligible_adjusters)
            print(f"     To: {new_adj['first_name']} {new_adj['last_name']} (specialist)")
            events.append(generate_claim_status_event(
                state, notes=f"Claim reassigned to {new_adj['first_name']} {new_adj['last_name']} after major loss escalation"
            ))
        else:
            print(f"     No eligible specialist adjusters available")
            events.append(generate_claim_status_event(
                state, notes="ALERT: Claim reassignment needed but no specialist adjusters available"
            ))

## Initialize Simulation State

In [ ]:
sim_states = []

for claim in active_claims:
    policy = policies.get(claim["policy_id"])
    policyholder = policyholders.get(claim["policyholder_id"])
    asset = insured_assets.get(claim["asset_id"])
    adjuster = adjusters.get(claim["adjuster_id"])
    office = offices.get(claim.get("office_id", ""))

    if not all([policy, policyholder, asset, adjuster]):
        continue
    if office is None:
        office = {"latitude": 0, "longitude": 0}

    state = ClaimSimState(claim, policy, policyholder, asset, adjuster, office)

    # Make some claims high-value for demo scenario variety
    if random.random() < 0.15:
        state.estimated_loss = random.uniform(100000, 500000)

    # Make some claims already past SLA for demo
    if random.random() < 0.10:
        state.filed_time = datetime.utcnow() - timedelta(days=random.randint(20, 45))

    sim_states.append(state)

print(f"\nInitialized {len(sim_states)} claim simulations")
for s in sim_states[:5]:
    print(f"  Claim {s.claim['claim_number']}: {s.claim.get('claim_type', 'N/A')}, "
          f"loss=${s.estimated_loss:,.0f}, priority={s.priority}, "
          f"adjuster={s.adjuster.get('last_name', 'N/A')}")
if len(sim_states) > 5:
    print(f"  ... and {len(sim_states) - 5} more")

## Run Event Generation Loop

In [ ]:
print(f"\n{'='*60}")
print(f"Starting event generation for {SIMULATION_DURATION_MINUTES} minutes")
print(f"Active claims: {len(sim_states)}")
print(f"Tick interval: {TICK_INTERVAL_SECONDS}s")
print(f"Scenario injection: {'ON' if SCENARIO_INJECTION_ENABLED else 'OFF'}")
print(f"Destination: Eventhouse ({KUSTO_DATABASE})")
print(f"{'='*60}\n")

total_events = 0
start_time = time.time()
end_time = start_time + (SIMULATION_DURATION_MINUTES * 60)
tick = 0

try:
    while time.time() < end_time:
        tick += 1
        tick_events = []

        for state in sim_states:
            if state.finalized:
                continue

            # Advance claim processing by one tick
            transitioned = state.advance_tick()

            # 1. ClaimStatusEvent on every transition
            if transitioned:
                tick_events.append(generate_claim_status_event(state))

                # 2. InspectionEvent when reaching site_inspection
                if state.current_status == "site_inspection":
                    tick_events.append(generate_inspection_event(state, "field_inspection", "scheduled"))

                # 3. PaymentEvent when payment is issued
                if state.current_status == "payment_issued":
                    tick_events.append(generate_payment_event(state))

            # 4. Random fraud alerts (~10% chance per tick for non-finalized claims)
            if random.random() < 0.10:
                tick_events.append(generate_fraud_alert_event(state))

            # 5. Occasional policy changes (~5% chance per tick)
            if random.random() < 0.05:
                tick_events.append(generate_policy_change_event(state))

            # 6. Inject demo scenarios
            if SCENARIO_INJECTION_ENABLED:
                check_and_inject_scenarios(state, tick_events)

        # Buffer all events for Kusto ingest
        for event in tick_events:
            if PRINT_EVENTS:
                print(f"  [{event['event_type']}] claim={event['body'].get('claim_id', event['body'].get('policy_id', 'N/A'))[:8]}... "
                      f"status={event['body'].get('new_status', event['body'].get('alert_type', 'N/A'))}")
            buffer_event(event)

        total_events += len(tick_events)

        # Progress update every 5 ticks
        if tick % 5 == 0:
            flush_all_buffers()
            elapsed = time.time() - start_time
            active_count = sum(1 for s in sim_states if not s.finalized)
            print(f"\n--- Tick {tick} | {elapsed:.0f}s elapsed | {total_events} events | {active_count} active claims ---")
            print(f"    Ingested: {dict(_ingest_counts)}")

        time.sleep(TICK_INTERVAL_SECONDS)

except KeyboardInterrupt:
    print("\n\nSimulation stopped by user.")

finally:
    flush_all_buffers()
    elapsed = time.time() - start_time
    finalized = sum(1 for s in sim_states if s.finalized)
    print(f"\n{'='*60}")
    print(f"Simulation complete")
    print(f"  Duration: {elapsed:.0f}s ({elapsed/60:.1f} min)")
    print(f"  Total events generated: {total_events}")
    print(f"  Claims finalized: {finalized}/{len(sim_states)}")
    print(f"  Events ingested per table:")
    for table, count in _ingest_counts.items():
        print(f"    {table:<30} {count:>6}")
    print(f"{'='*60}")

## Verify: Query Eventhouse Tables

In [ ]:
print("Eventhouse row counts (may take a few minutes for queued ingestion to complete):\n")
for table in TABLE_COLUMNS:
    try:
        result = kusto_client.execute(KUSTO_DATABASE, f"{table} | count")
        for row in result.primary_results[0]:
            print(f"  {table:<30} {row[0]:>6} rows")
    except Exception as e:
        print(f"  {table:<30} ERROR: {e}")

In [ ]:
# Sample: latest claim status events
print("\nLatest claim status transitions:")
query = """
ClaimStatusEvent
| summarize arg_max(timestamp, *) by claim_id
| project timestamp, claim_id, claim_number, previous_status, new_status, priority, notes
| order by timestamp desc
| take 10
"""
result = kusto_client.execute(KUSTO_DATABASE, query)
for row in result.primary_results[0]:
    print(f"  {row['timestamp']}  claim={str(row['claim_number'])}  "
          f"{row['previous_status']} -> {row['new_status']}  "
          f"priority={row['priority']}")

In [ ]:
# Sample: fraud alerts by severity
print("\nFraud alerts by severity:")
query = """
FraudAlertEvent
| summarize count() by severity, alert_type
| order by severity asc, count_ desc
"""
result = kusto_client.execute(KUSTO_DATABASE, query)
for row in result.primary_results[0]:
    print(f"  [{row['severity']}] {row['alert_type']}: {row['count_']} alerts")

## Event Summary

Events are ingested directly into Eventhouse tables via the Kusto Python SDK.
Queued ingestion may take 2-5 minutes to fully materialize.

### Insurance Event Types Generated
- **ClaimStatusEvent** — Claim lifecycle transitions through processing stages
- **FraudAlertEvent** — Automated fraud detection alerts with confidence scoring
- **InspectionEvent** — Asset inspection scheduling and completion
- **PolicyChangeEvent** — Policy modifications during claim processing
- **PaymentEvent** — Claim settlement payments and disbursements

### Scenario Trigger Summary
- **Late Resolution**: `ClaimStatusEvent | where notes has "SLA breach"`
- **Major Loss**: `FraudAlertEvent | where severity == "high"` + `ClaimStatusEvent | where notes has "Major loss"`
- **Adjuster Overload Risk**: `ClaimStatusEvent | where notes has "capacity"`
- **Inspection Due**: `InspectionEvent | where result == "scheduled"`
- **Claim Reassignment**: `ClaimStatusEvent | where notes has "reassigned"`